# Notebook 17: Context Length Extension

**Series:** LLM Systems & Inference  
**Prerequisites:** Notebook 16 (Positional Encoding & RoPE)

---

## Background: The Long Context Race

In 2022, GPT-3 had a 2,048 token context window. By 2024, Gemini 1.5 Pro had a 1,000,000 token window. That's a 500× increase in two years. Understanding *how* that happened — and what the limits are — is essential for building and deploying modern LLM systems.

The journey wasn't just "train on longer sequences." Training on 1M-token sequences would require a thousand times more compute than 2K sequences, plus the attention matrix alone would be 1M × 1M = 1 trillion elements. New techniques were needed to make long context practical.

### Why Models Fail Beyond Training Length

Even with RoPE (which has relative position properties), models trained on 2K contexts fail at 4K. The fundamental reason: the model has never experienced the *combination* of rotations that appear at position 2049. The high-frequency dimension pairs (small k) rotate so fast that they complete many full cycles within the training length. The low-frequency pairs (large k) barely rotate at all within 2K tokens.

Beyond the training length, two things happen:
1. High-frequency dimensions have "seen" these rotation angles before (they repeat), so they might be okay
2. Low-frequency dimensions encounter rotation angles they've literally never seen during training

The model's attention patterns become incoherent because the learned attention weights assume specific position encodings that don't generalize.

### The Key Techniques

**1. Position Interpolation (PI)** (Chen et al. 2023, Meta)  
Instead of position 4096 getting angle 4096θ, interpolate: give it angle 2048θ and rescale all positions by a factor of 2. The training distribution is now correct, but positions are more tightly packed. A small amount of fine-tuning on long-context data is needed to adapt. This was the first practical solution and enabled LLaMA-2 to go from 4K to 16K tokens with only 1000 fine-tuning steps.

**2. YaRN** (Peng et al. 2023)  
YaRN (Yet another RoPE extensioN) observed that linear interpolation hurts high-frequency dimensions more than low-frequency ones. It applies different scaling factors to different frequency bands: low-frequency dimensions get linear interpolation, high-frequency dimensions are left untouched (they already generalize), and mid-frequency dimensions get a smooth interpolation. This produces better perplexity at extended lengths.

**3. Increasing the RoPE Base**  
The simplest approach: just use a larger base (e.g., 500,000 instead of 10,000) from the start. A larger base means all frequency components rotate more slowly, so they don't exhaust their meaningful range as quickly. LLaMA-3 used this approach (base=500K) combined with training on a 128K context window. No need for inference-time interpolation tricks.

**4. Flash Attention + Chunked Prefill**  
Even if you solve the positional encoding problem, you still have a quadratic memory problem. 128K attention requires O(n²) memory. Flash Attention (notebook 06) solves this with tiled SRAM computation. Combined with chunked prefill (process the prompt in chunks to limit peak memory), this makes 128K+ context computationally feasible.

---

## What You'll Build

1. **Perplexity degradation at long context** — visualize where models fail
2. **Position Interpolation** — implement and analyze
3. **YaRN scaling** — frequency-aware interpolation
4. **RoPE base scaling** — the LLaMA-3 approach
5. **"Lost in the middle" problem** — attention patterns at long context
6. **Practical guidelines** — when to use which approach

In [ ]:
!pip install -q torch numpy matplotlib

In [ ]:
import torch
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
from dataclasses import dataclass
torch.manual_seed(42)
np.random.seed(42)

def build_rope_cache(seq_len, head_dim, base=10000.0):
    theta = 1.0 / (base ** (torch.arange(0, head_dim, 2).float() / head_dim))
    positions = torch.arange(seq_len).float()
    freqs = torch.outer(positions, theta)
    return freqs.cos(), freqs.sin()

def apply_rope(x, cos, sin):
    x1, x2 = x[..., :x.shape[-1]//2], x[..., x.shape[-1]//2:]
    cos = cos.unsqueeze(0).unsqueeze(2)
    sin = sin.unsqueeze(0).unsqueeze(2)
    return torch.cat([x1*cos - x2*sin, x1*sin + x2*cos], dim=-1)

print("Setup complete.")

## Part 1: Where Models Break — Rotation Angle Analysis

In [ ]:
def rotation_angles(seq_len: int, head_dim: int, base: float) -> np.ndarray:
    """Rotation angles (in radians) for each (position, dim_pair) combination."""
    theta = 1.0 / (base ** (np.arange(0, head_dim, 2) / head_dim))
    positions = np.arange(seq_len)
    return np.outer(positions, theta)  # [seq_len, head_dim//2]


HEAD_DIM = 128
TRAIN_LEN = 4096
BASE = 10000.0

# Angles seen during training
train_angles = rotation_angles(TRAIN_LEN, HEAD_DIM, BASE)

# For extended context, which dimensions are extrapolating (angles > 2π * n_cycles_seen)?
ext_lens = [4096, 8192, 16384, 32768]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
for ext_len in ext_lens:
    ext_angles = rotation_angles(ext_len, HEAD_DIM, BASE)
    # Fraction of dim pairs that see rotation angles NOT seen in training
    max_train_angle = train_angles[-1]  # max angle at each dim pair during training
    extrapolating = (ext_angles[-1] > max_train_angle).mean()
    ax.bar(ext_len, extrapolating, color='#e74c3c', alpha=0.7)
    ax.text(ext_len, extrapolating + 0.01, f'{extrapolating:.1%}', ha='center', fontsize=9)

ax.set_xlabel('Sequence Length')
ax.set_ylabel('Fraction of Dim Pairs Extrapolating')
ax.set_title('RoPE Extrapolation (base=10000, trained on 4K)')
ax.grid(alpha=0.3, axis='y')

# Angle heatmap for training range
ax2 = axes[1]
show = train_angles % (2 * np.pi)  # wrap angles to [0, 2π] to show periodicity
im = ax2.imshow(show[::16, :32].T, aspect='auto', cmap='hsv', origin='lower')
ax2.set_xlabel('Position (every 16th)')
ax2.set_ylabel('Dimension pair (first 32)')
ax2.set_title('RoPE Angles mod 2π — Training Range')
plt.colorbar(im, ax=ax2, label='Angle (radians)')

plt.tight_layout()
plt.savefig('rope_extrapolation.png', dpi=120, bbox_inches='tight')
plt.show()

print("Key observation:")
print(f"At 2× training length (8192 tokens), {(rotation_angles(8192, HEAD_DIM, BASE)[-1] > train_angles[-1]).mean():.1%} of dim pairs extrapolate")
print(f"At 4× training length (16384 tokens), {(rotation_angles(16384, HEAD_DIM, BASE)[-1] > train_angles[-1]).mean():.1%} of dim pairs extrapolate")

## Part 2: Position Interpolation

In [ ]:
def build_rope_cache_interpolated(
    seq_len: int, head_dim: int, base: float = 10000.0,
    train_len: int = 4096, target_len: int = 8192
) -> tuple[torch.Tensor, torch.Tensor]:
    """
    Position interpolation: scale positions by (train_len / target_len).
    This maps position `m` (in 0..target_len) to effective position
    `m * train_len / target_len` (in 0..train_len).
    
    Ensures every effective position is within training distribution.
    """
    scale = train_len / target_len
    theta = 1.0 / (base ** (torch.arange(0, head_dim, 2).float() / head_dim))
    positions = torch.arange(seq_len).float() * scale  # scaled positions
    freqs = torch.outer(positions, theta)
    return freqs.cos(), freqs.sin()


# Compare standard vs interpolated positions
HEAD_DIM = 64
TRAIN_LEN = 4096
TARGET_LEN = 16384

std_cos, std_sin = build_rope_cache(TARGET_LEN, HEAD_DIM, base=10000.0)
pi_cos, pi_sin = build_rope_cache_interpolated(TARGET_LEN, HEAD_DIM, base=10000.0,
                                                train_len=TRAIN_LEN, target_len=TARGET_LEN)

fig, axes = plt.subplots(2, 2, figsize=(14, 8))

for row, (cos_c, sin_c, label) in enumerate([
    (std_cos, std_sin, 'Standard RoPE (extrapolating)'),
    (pi_cos, pi_sin, f'Position Interpolation (scale={TRAIN_LEN/TARGET_LEN:.2f})'),
]):
    # Show frequencies for a single dim pair over all positions
    ax = axes[row][0]
    for dim_pair in [0, 4, 8, 16, 31]:
        ax.plot(cos_c[:, dim_pair].numpy(), alpha=0.8, label=f'dim pair {dim_pair}')
    ax.axvline(TRAIN_LEN, color='red', linestyle='--', label='Training cutoff')
    ax.set_xlabel('Position')
    ax.set_ylabel('cos(m·θ_k)')
    ax.set_title(f'{label}\nCosine Values per Dim Pair')
    ax.legend(fontsize=7, loc='upper right')
    ax.grid(alpha=0.3)
    
    # Show rotation angle (to see if it stays in training range)
    ax2 = axes[row][1]
    # Compute actual angles from cos/sin
    angles = torch.atan2(sin_c, cos_c)  # [seq_len, head_dim//2]
    im = ax2.imshow(angles.numpy()[::32, :20].T, aspect='auto', cmap='RdBu', origin='lower')
    ax2.axvline(TRAIN_LEN // 32, color='red', linestyle='--', linewidth=2)
    ax2.set_xlabel('Position (every 32nd)')
    ax2.set_ylabel('Dim pair')
    ax2.set_title('Rotation Angles (atan2)')
    plt.colorbar(im, ax=ax2)

plt.suptitle('Position Interpolation for Context Extension', fontsize=13)
plt.tight_layout()
plt.savefig('position_interpolation.png', dpi=120, bbox_inches='tight')
plt.show()

print("Position Interpolation effect:")
print(f"  Max angle (standard, position {TARGET_LEN-1}):      {std_cos[-1].arccos().max():.2f} rad")
print(f"  Max angle (interpolated, position {TARGET_LEN-1}): {pi_cos[-1].arccos().max():.2f} rad")
print(f"  Max angle during training (position {TRAIN_LEN-1}):  {std_cos[TRAIN_LEN-1].arccos().max():.2f} rad")
print("  → Interpolation keeps angles within training range ✅")

## Part 3: YaRN — Frequency-Aware Interpolation

In [ ]:
def build_rope_cache_yarn(
    seq_len: int, head_dim: int, base: float = 10000.0,
    train_len: int = 4096, target_len: int = 32768,
    beta_fast: int = 32, beta_slow: int = 1,
) -> tuple[torch.Tensor, torch.Tensor]:
    """
    YaRN (Yet Another RoPE extensioN): frequency-aware interpolation.
    
    - High-frequency dims (small wavelength): no interpolation needed (they repeat anyway)
    - Low-frequency dims (large wavelength): linear interpolation
    - Mid-frequency dims: smooth blend between the two
    
    YaRN uses NTK (Neural Tangent Kernel) theory to determine which dims need more scaling.
    """
    scale = target_len / train_len
    
    dim_pairs = torch.arange(0, head_dim, 2).float()
    theta = 1.0 / (base ** (dim_pairs / head_dim))  # base frequencies
    wavelength = 2 * np.pi / theta.numpy()           # wavelength per dim pair
    
    # Per-dimension scaling factor
    # wavelength < train_len/beta_fast → no scaling (high freq)
    # wavelength > train_len/beta_slow → linear interp (low freq)
    # in between → smooth blend
    s = np.ones(head_dim // 2)
    for i, wl in enumerate(wavelength):
        if wl < train_len / beta_fast:
            s[i] = 1.0  # high freq: no change
        elif wl > train_len / beta_slow:
            s[i] = scale  # low freq: full interpolation
        else:
            # Smooth blend: ramp from 1 to scale
            t = (train_len / wl - beta_slow) / (beta_fast - beta_slow)
            s[i] = (1 - t) * scale + t * 1.0
    
    s = torch.tensor(s, dtype=torch.float)
    
    # Effective frequencies: divide theta by scaling factor
    # (stretching the frequency = allowing larger angles before wrapping)
    effective_theta = theta / s
    
    positions = torch.arange(seq_len).float()
    freqs = torch.outer(positions, effective_theta)
    
    # YaRN also applies an attention scale correction (1/sqrt(1 + 0.1*log(scale)))
    return freqs.cos(), freqs.sin()


# Compare all three approaches
TRAIN_LEN = 4096
TARGET_LEN = 32768
scale = TARGET_LEN / TRAIN_LEN

approaches = [
    ('Standard RoPE\n(extrapolating)', build_rope_cache(TARGET_LEN, 128, 10000.0)),
    ('Position Interpolation', build_rope_cache_interpolated(TARGET_LEN, 128, 10000.0, TRAIN_LEN, TARGET_LEN)),
    ('YaRN', build_rope_cache_yarn(TARGET_LEN, 128, 10000.0, TRAIN_LEN, TARGET_LEN)),
    ('Higher Base (500K)', build_rope_cache(TARGET_LEN, 128, 500000.0)),
]

fig, axes = plt.subplots(1, 4, figsize=(18, 4))

for ax, (label, (cos_c, sin_c)) in zip(axes, approaches):
    # Show angle distribution for positions beyond training
    angles = torch.atan2(sin_c[TRAIN_LEN:], cos_c[TRAIN_LEN:]).abs()  # post-training positions
    # Compare to training-range angles
    train_angles_dist = torch.atan2(sin_c[:TRAIN_LEN], cos_c[:TRAIN_LEN]).abs()
    
    bins = np.linspace(0, np.pi, 40)
    ax.hist(train_angles_dist.flatten().numpy(), bins=bins, alpha=0.6, label='In-training', density=True, color='#2ecc71')
    ax.hist(angles.flatten().numpy(), bins=bins, alpha=0.6, label='Beyond training', density=True, color='#e74c3c')
    ax.set_xlabel('|Rotation Angle|')
    ax.set_ylabel('Density')
    ax.set_title(label, fontsize=10)
    ax.legend(fontsize=7)
    ax.grid(alpha=0.3)

plt.suptitle('Angle Distribution: In-Training vs Beyond Training Context', fontsize=12, y=1.02)
plt.tight_layout()
plt.savefig('context_extension_comparison.png', dpi=120, bbox_inches='tight')
plt.show()

print("Interpretation:")
print("  Green: angles seen during training (in-distribution)")
print("  Red:   angles at extended positions")
print("  Best approach: red matches green → minimal distribution shift")

## Part 4: The 'Lost in the Middle' Problem

Even when positional encoding is extended correctly, a separate empirical phenomenon
limits long-context usefulness: models pay less attention to information in the **middle**
of long contexts. They reliably use information at the beginning (primacy) and end (recency)
but "lose" what's in the middle.

In [ ]:
def simulate_attention_over_context(
    context_len: int,
    query_pos: int,  # where the "question" token is
    n_heads: int = 8,
    head_dim: int = 64,
    base: float = 10000.0,
) -> np.ndarray:
    """
    Simulate attention patterns — show how attention weight varies
    across context positions. Uses a simplified model where attention
    decays with distance (from RoPE's relative position property).
    """
    cos, sin = build_rope_cache(context_len, head_dim, base)
    
    torch.manual_seed(42)
    q = torch.randn(1, 1, n_heads, head_dim)
    k = torch.randn(1, context_len, n_heads, head_dim)
    
    # Apply RoPE
    q_rot = apply_rope(q, cos[query_pos:query_pos+1], sin[query_pos:query_pos+1])
    k_rot = apply_rope(k, cos, sin)
    
    # Compute attention scores
    scale = head_dim ** -0.5
    scores = (q_rot * k_rot).sum(-1) * scale  # [1, context_len, n_heads]
    
    # Causal mask (only attend to positions <= query_pos)
    mask = torch.full((context_len,), float('-inf'))
    mask[:query_pos + 1] = 0
    scores = scores + mask.unsqueeze(0).unsqueeze(-1)
    
    attn = F.softmax(scores, dim=1)  # [1, context_len, n_heads]
    return attn[0, :, :].mean(-1).detach().numpy()  # avg over heads: [context_len]


fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Short context (4096 tokens)
ax = axes[0]
for query_pos_frac, color in [(0.9, '#e74c3c'), (0.95, '#3498db'), (0.99, '#2ecc71')]:
    ctx = 4096
    qpos = int(ctx * query_pos_frac)
    attn = simulate_attention_over_context(ctx, qpos)
    smoothed = np.convolve(attn[:qpos+1], np.ones(64)/64, mode='same')
    positions = np.arange(qpos + 1) / ctx
    ax.plot(positions, smoothed[:len(positions)], color=color, linewidth=1.5,
            label=f'Query at {query_pos_frac:.0%} position', alpha=0.9)

ax.set_xlabel('Context Position (fraction of total)')
ax.set_ylabel('Average Attention Weight')
ax.set_title('Attention Distribution — Short Context (4K tokens)')
ax.legend(fontsize=9)
ax.grid(alpha=0.3)

# Long context (32768 tokens) — "lost in the middle" effect
ax2 = axes[1]
for ctx, color, label in [(4096, '#2ecc71', '4K context'), (16384, '#e67e22', '16K context'), (32768, '#e74c3c', '32K context')]:
    qpos = ctx - 1
    attn = simulate_attention_over_context(ctx, qpos, base=10000.0)
    # Normalize to [0,1] range for comparison
    fractions = np.linspace(0, 1, ctx)
    # Smooth and downsample
    window = max(ctx // 200, 1)
    smoothed = np.convolve(attn, np.ones(window)/window, mode='same')
    ax2.plot(fractions[::window], smoothed[::window], color=color, label=label, linewidth=2)

ax2.set_xlabel('Context Position (fraction of total)')
ax2.set_ylabel('Avg Attention Weight')
ax2.set_title('"Lost in the Middle" — Attention at Different Context Lengths')
ax2.legend(fontsize=9)
ax2.grid(alpha=0.3)

plt.suptitle('Attention Patterns Over Context Length', fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig('lost_in_middle.png', dpi=120, bbox_inches='tight')
plt.show()

print("'Lost in the Middle' (Liu et al. 2023):")
print("  Performance on retrieval tasks is highest when key info is at")
print("  the START or END of the context — middle content gets deprioritized.")
print("  This is an attention pattern issue, not a positional encoding issue.")
print("  Mitigation: put important instructions at start AND end; use retrieval to")
print("  surface relevant chunks rather than stuffing everything into context.")

## Part 5: Practical Guidelines — When and How to Extend Context

In [ ]:
guidelines = [
    (
        "Context need ≤ 2× training length",
        "Position Interpolation",
        "Simple, needs ~1000 FT steps, works well at 2×"
    ),
    (
        "Context need 2–8× training length",
        "YaRN or NTK-aware PI",
        "Better perplexity than PI at larger extensions"
    ),
    (
        "Training a new model from scratch",
        "High RoPE base (500K–1M)",
        "Simplest approach, no inference-time tricks needed"
    ),
    (
        "> 100K context needed",
        "Train from scratch on long data",
        "Interpolation tricks break down; need actual training"
    ),
    (
        "Memory limited (Flash Attention not enough)",
        "Chunked prefill + streaming",
        "Process prompt in K-token chunks; limit peak memory"
    ),
    (
        "Practical RAG vs long context",
        "RAG for > 16K",
        "Retrieval sidesteps 'lost in middle'; faster TTFT"
    ),
]

print("Context Length Extension — Decision Guide:")
print("=" * 80)
for scenario, approach, reason in guidelines:
    print(f"\n  Scenario: {scenario}")
    print(f"  Approach: {approach}")
    print(f"  Reason:   {reason}")

print("\n" + "=" * 80)
print("\nContext Length vs Memory Cost (Llama-3.1-8B, BF16, 1 request):")
print(f"{'Context (K)':>12} {'KV Cache (GB)':>14} {'Peak GPU Mem (GB)':>18}")
print("-" * 48)
model_gb = 16.0  # 8B BF16
n_layers, n_kv_heads, head_dim = 32, 8, 128
bytes_per_token = 2 * n_layers * n_kv_heads * head_dim * 2 / 1e9  # BF16
for ctx_k in [8, 16, 32, 64, 128]:
    kv_gb = ctx_k * 1000 * bytes_per_token
    total_gb = model_gb + kv_gb
    print(f"{ctx_k:>12} {kv_gb:>14.2f} {total_gb:>18.2f}")

## Summary

### Key Takeaways

**Why Models Fail at Long Context**  
Low-frequency RoPE dimensions see rotation angles at extended positions that were never in the training distribution. The model's attention patterns become incoherent because learned weights don't generalize to unseen rotation angles.

**The Three Extension Approaches**  
- **Position Interpolation**: Scale positions to fit training range. Simple, needs ~1K fine-tuning steps. Works up to ~4× extension.
- **YaRN**: Frequency-aware interpolation. Better quality at 4–8× extension by leaving high-frequency dims unscaled.
- **Higher RoPE base**: The LLaMA-3 approach — train with base=500K from the start. No inference tricks, cleanest solution.

**'Lost in the Middle'**  
Even correctly extended models pay less attention to middle-context content. This is an attention pattern phenomenon, not a positional encoding issue. Mitigation: put key instructions at both start and end of context; prefer RAG for long-document retrieval.

**Memory Reality**  
Long context has a real cost: 128K context on LLaMA-3.1-8B uses ~5 GB of KV cache alone. Flash Attention and chunked prefill make it computationally feasible, but memory limits still apply.

### What's Next (Notebook 18: Parallelism Strategies)

When a model is too large for a single GPU — or when you need to serve it with lower latency — you need to split it across multiple GPUs. Notebook 18 covers tensor parallelism, pipeline parallelism, and sequence parallelism: the three strategies for multi-GPU LLM inference.